- Thông tin thêm về data: https://huggingface.co/datasets/uitnlp/vietnamese_students_feedback
- Github data: https://github.com/buyamm/Sentiment-Analysis-VKU

In [1]:
import pandas as pd
import re
from nltk import ngrams

In [2]:
df = pd.read_csv("/kaggle/input/datasets/thngonquang/students-feedback/vietnamese_students_feedback_train.csv").dropna()
df_test = pd.read_csv("/kaggle/input/datasets/thngonquang/students-feedback/vietnamese_students_feedback_test.csv").dropna()
df_validation = pd.read_csv("/kaggle/input/datasets/thngonquang/students-feedback/vietnamese_students_feedback_validation.csv").dropna()

In [3]:
print(len(df))
print(len(df_test))
print(len(df_validation))

11426
3166
1583


In [4]:
print(df.head(5))
print(df_test.head(5))
print(df_validation.head(5))

                                            sentence  sentiment  topic
0                          slide giáo trình đầy đủ .          2      1
1     nhiệt tình giảng dạy , gần gũi với sinh viên .          2      0
2               đi học đầy đủ full điểm chuyên cần .          0      1
3  chưa áp dụng công nghệ thông tin và các thiết ...          0      0
4  thầy giảng bài hay , có nhiều bài tập ví dụ ng...          2      0
                                            sentence  sentiment  topic
0                           nói tiếng anh lưu loát .          2      0
1                           giáo viên rất vui tính .          2      0
2                                    cô max có tâm .          2      0
3                       giảng bài thu hút , dí dỏm .          2      0
4  giáo viên không giảng dạy kiến thức , hướng dẫ...          0      0
                                            sentence  sentiment  topic
0                           giáo trình chưa cụ thể .          0      1
1     

In [5]:
print("sentiment", df['sentiment'].unique())
print("topic: ", df['topic'].unique())

sentiment [2 0 1]
topic:  [1 0 3 2]


In [6]:
print("sentiment", df['sentiment'].value_counts())
print("topic: ", df['topic'].value_counts())

# sentiment: 0 (negative), 1 (neutral) and 2 (positive).
# topic: 0 (lecturer), 1 (training_program), 2 (facility) and 3 (others).

sentiment sentiment
2    5643
0    5325
1     458
Name: count, dtype: int64
topic:  topic
0    8166
1    2201
3     562
2     497
Name: count, dtype: int64


In [7]:
# Độ dài ký tự của mỗi dòng
df['length'] = df['sentence'].astype(str).apply(len)

# Min, Max
print("Min:", df['length'].min())
print("Max:", df['length'].max())

Min: 4
Max: 660


In [8]:
df['sentence'].str.len().describe()

count    11426.000000
mean        59.084894
std         43.085202
min          4.000000
25%         31.000000
50%         47.000000
75%         73.000000
max        660.000000
Name: sentence, dtype: float64

In [9]:
# df_sample.to_csv("viewdata.csv", sep='\t', encoding='utf-8')

# Làm sạch data

In [10]:
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 88.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.2 MB/s eta 0:00:00


In [11]:
import unicodedata
import re
from underthesea import word_tokenize
stopwords = open('/kaggle/input/datasets/thngonquang/vietnamese-stopwords/vietnamese-stopwords-dash.txt', encoding='utf-8').read().splitlines()

In [12]:
# stopwords

In [13]:
def preprocess(text):
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    
    # slang normalize
    slang_dict = {
        "gv": "giảng viên",
        "sv": "sinh viên",
        "hk": "học kỳ"
    }
    for k, v in slang_dict.items():
        text = text.replace(k, v)
    
    # giữ emoji cơ bản
    text = re.sub(r'[^0-9a-zA-ZÀ-ỹ\s:()!?.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    text = word_tokenize(text, format="text")
    
    return text

In [14]:
X_train = df['sentence']
X_test = df_test['sentence']
X_validation = df_validation['sentence']

y_train = df['sentiment']
y_test = df_test['sentiment']
y_validation = df_validation['sentiment']

In [15]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)
X_validation = X_validation.apply(preprocess)

In [16]:
print(X_train.head(5))
print(X_test.head(5))
print(X_validation.head(5))

0                            slide giáo_trình đầy_đủ .
1         nhiệt_tình giảng_dạy gần_gũi với sinh_viên .
2                 đi học đầy_đủ full_điểm chuyên cần .
3    chưa áp_dụng công_nghệ_thông_tin và các thiết_...
4    thầy giảng bài hay có nhiều bài_tập ví_dụ ngay...
Name: sentence, dtype: object
0                             nói tiếng anh lưu_loát .
1                             giáo_viên rất vui_tính .
2                                      cô max có tâm .
3                           giảng bài thu_hút dí_dỏm .
4    giáo_viên không giảng_dạy kiến_thức hướng_dẫn ...
Name: sentence, dtype: object
0                             giáo_trình chưa cụ_thể .
1                                     giảng buồn_ngủ .
2                         giáo_viên vui_tính tận_tâm .
3    giảng_viên nên giao bài_tập nhiều hơn chia nhó...
4    giảng_viên cần giảng bài chi_tiết hơn đi_sâu h...
Name: sentence, dtype: object


# **TFIDF**

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix 

tfidf = TfidfVectorizer(ngram_range=(1, 2))
X_train_vectorizer_tfidf = tfidf.fit_transform(X_train).toarray()
X_test_vectorizer = tfidf.transform(X_test).toarray()

## **MultinomialNB**

In [18]:
from sklearn.naive_bayes import MultinomialNB
# model
model = MultinomialNB()
model.fit(X_train_vectorizer_tfidf, y_train)

# Predict
y_pred_nb = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Độ chính xác của mô hình: 0.8682880606443462
              precision    recall  f1-score   support

           0       0.82      0.95      0.88      1409
           1       0.00      0.00      0.00       167
           2       0.91      0.88      0.90      1590

    accuracy                           0.87      3166
   macro avg       0.58      0.61      0.59      3166
weighted avg       0.83      0.87      0.85      3166

[[1343    0   66]
 [ 102    0   65]
 [ 184    0 1406]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

## **LogisticRegression**

In [19]:
from sklearn.linear_model import LogisticRegression

# model
model = LogisticRegression()
model.fit(X_train_vectorizer_tfidf, y_train)

# Predict
y_pred_nb = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Độ chính xác của mô hình: 0.8885028427037271
              precision    recall  f1-score   support

           0       0.85      0.97      0.91      1409
           1       0.69      0.05      0.10       167
           2       0.93      0.91      0.92      1590

    accuracy                           0.89      3166
   macro avg       0.82      0.64      0.64      3166
weighted avg       0.88      0.89      0.87      3166

[[1361    1   47]
 [  92    9   66]
 [ 144    3 1443]]


## **SVM**

In [20]:
from sklearn.svm import LinearSVC

# model
model = LinearSVC()
model.fit(X_train_vectorizer_tfidf, y_train)

# Predict
y_pred_nb = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Độ chính xác của mô hình: 0.8960833859759949
              precision    recall  f1-score   support

           0       0.87      0.96      0.91      1409
           1       0.60      0.14      0.23       167
           2       0.93      0.92      0.92      1590

    accuracy                           0.90      3166
   macro avg       0.80      0.67      0.69      3166
weighted avg       0.89      0.90      0.88      3166

[[1355    6   48]
 [  76   24   67]
 [ 122   10 1458]]


# **BoW**

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(ngram_range = (1,2))
X_train_vectorizer_bow = bow.fit_transform(X_train).toarray()
X_test_vectorizer = bow.transform(X_test).toarray()

## **MultinomialNB**

In [22]:
from sklearn.naive_bayes import MultinomialNB
# model
model = MultinomialNB()
model.fit(X_train_vectorizer_bow, y_train)

# Predict
y_pred_nb = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Độ chính xác của mô hình: 0.8733417561591914
              precision    recall  f1-score   support

           0       0.83      0.96      0.89      1409
           1       0.67      0.01      0.02       167
           2       0.92      0.89      0.90      1590

    accuracy                           0.87      3166
   macro avg       0.81      0.62      0.61      3166
weighted avg       0.87      0.87      0.85      3166

[[1346    0   63]
 [ 101    2   64]
 [ 172    1 1417]]


## **LogisticRegression**

In [23]:
from sklearn.linear_model import LogisticRegression

# model
model = LogisticRegression()
model.fit(X_train_vectorizer_bow, y_train)

# Predict
y_pred_nb = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Độ chính xác của mô hình: 0.8932406822488945
              precision    recall  f1-score   support

           0       0.88      0.94      0.91      1409
           1       0.64      0.21      0.32       167
           2       0.91      0.92      0.92      1590

    accuracy                           0.89      3166
   macro avg       0.81      0.69      0.71      3166
weighted avg       0.88      0.89      0.88      3166

[[1325   10   74]
 [  61   35   71]
 [ 112   10 1468]]


## **SVM**

In [24]:
from sklearn.svm import LinearSVC

# model
model = LinearSVC()
model.fit(X_train_vectorizer_bow, y_train)

# Predict
y_pred_nb = model.predict(X_test_vectorizer)

accuracy = accuracy_score(y_test, y_pred_nb)
print(f"Độ chính xác của mô hình: {accuracy}")

nb_report = classification_report(y_test, y_pred_nb, output_dict = True)
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

Độ chính xác của mô hình: 0.8856601389766267
              precision    recall  f1-score   support

           0       0.90      0.92      0.91      1409
           1       0.45      0.34      0.39       167
           2       0.91      0.92      0.91      1590

    accuracy                           0.89      3166
   macro avg       0.75      0.72      0.74      3166
weighted avg       0.88      0.89      0.88      3166

[[1292   37   80]
 [  49   57   61]
 [ 102   33 1455]]
